# Imports

In [5]:
import logging, warnings; logging.getLogger().setLevel(logging.ERROR);
warnings.filterwarnings("ignore")

import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import re
from pathlib import Path 
import os

import warnings, scipy.sparse as sp, matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.pyplot import rc_context
import matplotlib.font_manager
import matplotlib.lines as lines

import gseapy as gp 

pd.set_option('display.max_rows', 200)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rc('font', size=12)

sc.settings.n_jobs=-1
sc.set_figure_params(dpi=80, dpi_save=300, color_map='Spectral_r', vector_friendly=True, transparent=True)
sc.settings.figdir = '../../1_outputs/0_figures'
sc.settings.verbosity = 1 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

%matplotlib inline 
%config InlineBackend.figure_format = 'retina'

In [13]:
def convert_xlsx_to_rnk_per_sheet(input_xlsx, out_dir):
    xls = pd.ExcelFile(input_xlsx)

    for sheet in xls.sheet_names:
        df = pd.read_excel(input_xlsx, sheet_name=sheet, usecols=[0, 1])
        df.columns = ["#gene", "#scores"]

        out_rnk = os.path.join(out_dir, f"{sheet}.rnk")
        df.to_csv(out_rnk, sep="\t", index=False, header=True)

In [14]:
convert_xlsx_to_rnk_per_sheet(
    input_xlsx="/Users/mkaur/github/2_tff1/1_tecs/1_pyzone/1_outputs/1_degs/ko_vs_wt_by_celltype.xlsx",
    out_dir="/Users/mkaur/github/2_tff1/1_tecs/1_pyzone/1_outputs/2_files/1_rnk/"
)

In [12]:
gp.get_library_name()[:11]

['ARCHS4_Cell-lines',
 'ARCHS4_IDG_Coexp',
 'ARCHS4_Kinases_Coexp',
 'ARCHS4_TFs_Coexp',
 'ARCHS4_Tissues',
 'Achilles_fitness_decrease',
 'Achilles_fitness_increase',
 'Aging_Perturbations_from_GEO_down',
 'Aging_Perturbations_from_GEO_up',
 'Allen_Brain_Atlas_10x_scRNA_2021',
 'Allen_Brain_Atlas_down']

In [ ]:
Iterate through the different pathways and through all of the cell types 

systematically 



In [ ]:
# Run preranked GSEA
pre_res1 = gp.prerank(
    rnk='allTEC_DEGs_scores.rnk',           # or pass a DataFrame instead
    gene_sets='KEGG_2019_Mouse',         # or 'GO_Biological_Process_2021', etc.
    outdir='/Users/mkaur/github/2_tff1/1_tecs/1_pyzone/1_outputs/3_gsea',                # output folder
    min_size=15,
    max_size=500,
    permutation_num=1000,                # use 1000+ for final results
    seed=42,
    format='png'
)

In [16]:
rnk_dir = Path("/Users/mkaur/github/2_tff1/1_tecs/1_pyzone/1_outputs/2_files/1_rnk")          # folder with *.rnk
out_dir= Path("/Users/mkaur/github/2_tff1/1_tecs/1_pyzone/1_outputs/3_gsea")     # where to write results
out_dir.mkdir(parents=True, exist_ok=True)

In [17]:
databases = [
    "KEGG_2019_Mouse",
    "WikiPathways_2024_Mouse",
    "Reactome_Pathways_2024",
    "MSigDB_Hallmark_2020",
]

In [18]:
rnk_files = sorted(rnk_dir.glob("*.rnk"))

for rnk_path in rnk_files:
    celltype = rnk_path.stem
    print(f"\n=== {celltype} ===")

    rnk = pd.read_csv(rnk_path, sep=None, engine="python", header=None)
    rnk = rnk.iloc[:, :2]
    rnk.columns = ["gene", "score"]
    rnk = rnk.dropna()
    rnk["score"] = pd.to_numeric(rnk["score"], errors="coerce")
    rnk = rnk.dropna()
    rnk = rnk.groupby("gene", as_index=False)["score"].max()
    rnk = rnk.sort_values("score", ascending=False)

    # one excel per rnk, one sheet per db
    excel_path = out_dir / f"{celltype}_gsea.xlsx"

    with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
        for db in databases:
            outdir = out_dir / celltype / db
            outdir.mkdir(parents=True, exist_ok=True)

            pre_res = gp.prerank(
                rnk=rnk,                 # dataframe works
                gene_sets=db,            # Enrichr library name
                outdir=str(outdir),      # gseapy will also save its own outputs here
                permutation_num=1000,     # bump later (e.g., 1000) once pipeline works
                min_size=15,
                max_size=500,
                seed=42,
                verbose=False,
            )

            # results table -> save to a sheet + also as a tsv in that folder
            res = pre_res.res2d.copy()
            res.to_csv(outdir / "gsea_results.tsv", sep="\t", index=False)

            # Excel sheet names must be <= 31 chars
            sheet = db[:31]
            res.to_excel(writer, sheet_name=sheet, index=False)

    print(f"Saved: {excel_path}")

2026-01-23 13:11:17,339 [WARNING] Duplicated values found in preranked stats: 50.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.



=== 0_cTEC ===


2026-01-23 13:11:40,747 [WARNING] Duplicated values found in preranked stats: 50.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:11:48,455 [WARNING] Duplicated values found in preranked stats: 50.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:12:32,927 [WARNING] Duplicated values found in preranked stats: 50.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:12:43,976 [WARNING] Duplicated values found in preranked stats: 99.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Saved: /Users/mkaur/github/2_tff1/1_tecs/1_pyzone/1_outputs/3_gsea/0_cTEC_gsea.xlsx

=== 10_mimetic(parathyroid) ===


2026-01-23 13:12:52,906 [WARNING] Duplicated values found in preranked stats: 99.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:12:57,059 [WARNING] Duplicated values found in preranked stats: 99.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:13:26,415 [WARNING] Duplicated values found in preranked stats: 99.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:13:32,704 [WARNING] Duplicated values found in preranked stats: 56.74% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


...

Saved: /Users/mkaur/github/2_tff1/1_tecs/1_pyzone/1_outputs/3_gsea/7_mimetic(neuroendo)_gsea.xlsx

=== 8_mimetic(ionocyte) ===


2026-01-23 13:26:34,613 [WARNING] Duplicated values found in preranked stats: 98.27% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:26:38,623 [WARNING] Duplicated values found in preranked stats: 98.27% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:27:07,509 [WARNING] Duplicated values found in preranked stats: 98.27% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:27:15,160 [WARNING] Duplicated values found in preranked stats: 97.41% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Saved: /Users/mkaur/github/2_tff1/1_tecs/1_pyzone/1_outputs/3_gsea/8_mimetic(ionocyte)_gsea.xlsx

=== 9_mimetic(ciliated) ===


2026-01-23 13:27:22,692 [WARNING] Duplicated values found in preranked stats: 97.41% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:27:26,798 [WARNING] Duplicated values found in preranked stats: 97.41% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-23 13:27:54,850 [WARNING] Duplicated values found in preranked stats: 97.41% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Saved: /Users/mkaur/github/2_tff1/1_tecs/1_pyzone/1_outputs/3_gsea/9_mimetic(ciliated)_gsea.xlsx
